In [ ]:
!pip install -q google-genai

In [ ]:
from IPython.display import display, Image, Markdown
from pprint import pprint

def print_response(response):
  print(f"Response id: {response.response_id}")
  print(f"Input tokens: {response.usage_metadata.prompt_token_count}; Output tokens: {response.usage_metadata.candidates_token_count}")

  for i in range(len(response.candidates)):
      display(Markdown(f"### Candidate #{i + 1}"))
      for part in response.candidates[i].content.parts:
          if part.text is not None:
              if part.thought:
                  markdown_content = f"<br/> <i>[Thinking...]</i> <br/> <i>{part.text}</i> <br/>"
              else:
                  markdown_content = f"<br/> <strong>Final response:</string> <br/> {part.text} <br/>"
              display(Markdown(markdown_content))

          if part.inline_data is not None:
              display(Image(data=part.inline_data.data, format="png"))

In [ ]:
from google import genai
from google.genai import types
from google.colab import userdata

api_key = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=api_key)

In [ ]:
generate_portrait_prompt = "A photorealistic portrait of a software developer. He is 24-years-old, from Italy, and wears glasses." \
                           "The scene is illuminated by the sunlight streaming through the open window, creating a welcoming and calm atmosphere." \
                           "Through the window a beautiful garden can be seen with all kinds of fruit trees and vegetables" \
                           "The overall mood is serene and masterful. Vertical portrait orientation."

edit_prompt = "Edit the portrait: The software developer is now looking dreamily through the window with a soft, absent-minded gaze, slightly turned away from the camera." \
              "He is holding a fork in one hand, mid-twirl, eating a plate of pasta bolognese placed on his desk next to the keyboard." \
              "His expression is wistful and relaxed, as if lost in thought while enjoying his meal." \
              "Keep everything else — the lighting, the home office setting, the warm tones, and the photorealistic style — exactly as before."

In [ ]:
generate_portrait_response = client.models.generate_content(
    model="gemini-3.1-flash-image-preview",
    contents=[generate_portrait_prompt],
    config=types.GenerateContentConfig(
        response_modalities=["Text", "Image"],
        thinking_config=types.ThinkingConfig(
            thinking_level="High",
            include_thoughts=True
        ),
        image_config=types.ImageConfig(
            aspect_ratio="16:9",
            image_size="512px"
        )
    )
)

In [ ]:
print_response(generate_portrait_response)

In [ ]:
conversation = [
    types.Content(role="user", parts=[types.Part(text=generate_portrait_prompt)]),
    types.Content(role="model", parts=[p for p in generate_portrait_response.candidates[0].content.parts if not p.thought]),
    types.Content(role="user", parts=[types.Part(text=edit_prompt)])
]

In [ ]:
make_edits_response = client.models.generate_content(
    model="gemini-3.1-flash-image-preview",
    contents=conversation,
    config=types.GenerateContentConfig(
        response_modalities=["Text", "Image"],
        thinking_config=types.ThinkingConfig(
            thinking_level="Minimal",
            include_thoughts=True
        ),
        image_config=types.ImageConfig(
            aspect_ratio="16:9",
            image_size="512px"
        )
    )
)

In [ ]:
print_response(make_edits_response)